In [1]:
%load_ext autoreload
%autoreload 2

from config import SimConfig, HybridSimConfig
from src.plants.fc_only_plant import FuelCellOnlyPlant
from src.plants.hybrid_plant import FuelCellBatteryPlant
from src.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker
from src.plotting import plot_dashboard

# Initialize Configs and Plants
cfg_base = SimConfig()
plant_base = FuelCellOnlyPlant(cfg_base)

cfg_hybrid = HybridSimConfig(lambda_scale=60, dt=5.0)
plant_hybrid = FuelCellBatteryPlant(cfg_hybrid)

# Cache data to RAM once
fleet_cache = load_and_cache_entire_fleet(cfg_base)

# Initialize our new Orchestrator (Excluding bad sensor days 1, 2, and 3)
benchmarker = VoyageBenchmarker(fleet_cache, exclude_days=[1, 2, 3])

Beginning memory staging of all 14 fleet files into RAM...
 -> Day 01 successfully cached in RAM.
 -> Day 02 successfully cached in RAM.
 -> Day 03 successfully cached in RAM.
 -> Day 04 successfully cached in RAM.
 -> Day 05 successfully cached in RAM.
 -> Day 06 successfully cached in RAM.
 -> Day 07 successfully cached in RAM.
 -> Day 08 successfully cached in RAM.
 -> Day 09 successfully cached in RAM.
 -> Day 10 successfully cached in RAM.
 -> Day 11 successfully cached in RAM.
 -> Day 12 successfully cached in RAM.
 -> Day 13 successfully cached in RAM.
 -> Day 14 successfully cached in RAM.

All 14 operational days securely held in RAM. Disk I/O locked.


In [2]:
# Create declarative definitions for whatever you want to test
baseline_heuristic = {
    "name": "Baseline Heuristic", "is_hybrid": False, "strategy": "HEURISTIC",
    "config": cfg_base, "plant": plant_base
}

baseline_sdp = {
    "name": "Baseline SDP", "is_hybrid": False, "strategy": "SDP",
    "config": cfg_base, "plant": plant_base
}

hybrid_heuristic = {
    "name": "Hybrid Heuristic", "is_hybrid": True, "strategy": "HEURISTIC",
    "config": cfg_hybrid, "plant": plant_hybrid
}

hybrid_tensor_sdp = {
    "name": "Hybrid Tensor SDP", "is_hybrid": True, "strategy": "SDP",
    "sdp_variant": "TENSOR_SWEEP", "config": cfg_hybrid, "plant": plant_hybrid
}

hybrid_mean_sdp = {
    "name": "Hybrid Mean SDP", "is_hybrid": True, "strategy": "SDP",
    "sdp_variant": "MEAN_PROXY", "config": cfg_hybrid, "plant": plant_hybrid
}

In [11]:
# Compare multiple strategies head-to-head
approaches_to_compare = {
    "Baseline Heuristic": baseline_heuristic,
    "Baseline Optimized": baseline_sdp,
    "Hybrid Heuristic": hybrid_heuristic,
    "Hybrid Optimized (Tensor)": hybrid_tensor_sdp,
    "Hybrid Optimized (Mean)": hybrid_mean_sdp
}

# Train on Days 4 through 13, Test on Day 14
df_comparison, sims = benchmarker.compare_approaches(
    approaches_to_compare, 
    train_days=[4, 5, 6, 7, 8, 9, 10, 12, 13, 14], 
    test_day=11
)

display(df_comparison)


Comparing 5 approaches | Train: [4, 5, 6, 7, 8, 9, 10, 12, 13, 14] | Test: Day 11
 -> Running: Baseline Heuristic
 -> Running: Baseline Optimized
 -> Running: Hybrid Heuristic
 -> Running: Hybrid Optimized (Tensor)
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (50 paths)...
 -> Tensors Cached. Initiating O(M^2) Online Sweep...
 -> Running: Hybrid Optimized (Mean)
 -> Launching MEAN_PROXY Solver...


,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s)
Baseline Heuristic,95.690761,31.690761,64.0,0.000000,NaN,0.003001
Baseline Optimized,188.904549,70.904549,118.0,0.000000,NaN,0.064144
Hybrid Heuristic,443.327191,265.595953,79.0,98.731237,49.228937,0.011520
Hybrid Optimized (Tensor),305.968334,67.560731,124.0,114.407603,53.560523,31.815821
Hybrid Optimized (Mean),283.129051,29.909102,155.0,98.219950,51.405827,56.036530


In [12]:
# Plot the Baseline SDP
# plot_dashboard(sims['Baseline Heuristic'], 'Baseline Heuristic', test_day=14, layout='grid')
# plot_dashboard(sims['Baseline Optimized'], 'Baseline Optimized', test_day=14, layout='grid')

# Plot the Hybrid comparison dashboard for the best performing approach
# plot_dashboard(sims["Hybrid Heuristic"], "Hybrid Heuristic", test_day=14, layout='grid')
plot_dashboard(sims["Hybrid Optimized (Mean)"], "Hybrid Optimized (Mean)", test_day=11, layout='grid')
plot_dashboard(sims["Hybrid Optimized (Tensor)"], "Hybrid Optimized (Tensor)", test_day=11, layout='grid')


In [5]:
# Run Chronological Forward Chaining for the Hybrid Mean SDP
print("--- APPROACH B: CHRONOLOGICAL FORWARD CHAINING ---")
df_forward = benchmarker.run_forward_chaining(hybrid_mean_sdp, min_train_days=1)
display(df_forward)

# Run Leave-One-Out for the Baseline SDP
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo = benchmarker.run_leave_one_out(hybrid_mean_sdp)
display(df_loo)

# You can now plot df_forward['Total Cost ($)'] just like you did in your original notebook!

--- APPROACH B: CHRONOLOGICAL FORWARD CHAINING ---

Starting Forward Chaining CV for: Hybrid Mean SDP
 -> Chaining Step 1: Training on [4] | Testing on 5
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 2: Training on [4, 5] | Testing on 6
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 3: Training on [4, 5, 6] | Testing on 7
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 4: Training on [4, 5, 6, 7] | Testing on 8
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 5: Training on [4, 5, 6, 7, 8] | Testing on 9
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 6: Training on [4, 5, 6, 7, 8, 9] | Testing on 10
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 7: Training on [4, 5, 6, 7, 8, 9, 10] | Testing on 11
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 8: Training on [4, 5, 6, 7, 8, 9, 10, 11] | Testing on 12
 -> Launching MEAN_PROXY Solver...
 -> Chaining Step 9: Training on [4, 5, 6, 7, 8, 9, 10, 11, 12] | Testing on 13
 -> Launching MEAN_PROXY Solver.

,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s),simulator,Train Horizon
Test Day 05,1.427794e+02,8.293223,70.0,6.448613e+01,53.379802,61.711678,<src.simulator.HybridSimulator object at 0x000...,1 days
Test Day 06,2.325280e+02,4.595072,137.0,9.093294e+01,57.268499,88.106268,<src.simulator.HybridSimulator object at 0x000...,2 days
Test Day 07,3.018585e+02,80.321788,126.0,9.553676e+01,52.288927,76.137798,<src.simulator.HybridSimulator object at 0x000...,3 days
Test Day 08,2.606101e+02,56.852021,122.0,8.175805e+01,50.992429,77.535542,<src.simulator.HybridSimulator object at 0x000...,4 days
Test Day 09,2.586564e+02,40.489292,126.0,9.216708e+01,52.696334,67.016542,<src.simulator.HybridSimulator object at 0x000...,5 days
Test Day 10,2.291100e+02,40.456096,105.0,8.365386e+01,50.679567,65.576920,<src.simulator.HybridSimulator object at 0x000...,6 days
Test Day 11,3.151009e+02,52.692413,161.0,1.014084e+02,52.618083,66.120533,<src.simulator.HybridSimulator object at 0x000...,7 days
Test Day 12,2.050000e+14,85638.711967,5.0,2.050000e+14,-128.236119,69.865487,<src.simulator.HybridSimulator object at 0x000...,8 days
Test Day 13,2.627229e+02,36.244540,133.0,9.347837e+01,54.271348,70.791272,<src.simulator.HybridSimulator object at 0x000...,9 days
Test Day 14,2.173122e+02,5.482429,114.0,9.782977e+01,53.818378,68.728167,<src.simulator.HybridSimulator object at 0x000...,10 days



--- APPROACH A: LEAVE ONE OUT ---

Starting Leave-One-Out CV for: Hybrid Mean SDP
 -> LOO Fold: Testing on 4 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 5 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 6 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 7 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 8 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 9 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 10 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 11 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 12 | Training on remaining 10 days
 -> Launching MEAN_PROXY Solver...
 -> LOO Fold: Testing on 13 | Training on remaining 10 days
 -> Laun

,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s),simulator
Test Day 04,4.169803e+02,120.340577,181.0,1.156397e+02,50.376543,65.384951,<src.simulator.HybridSimulator object at 0x000...
Test Day 05,2.110000e+14,85705.792283,4.0,2.110000e+14,-143.904345,62.315905,<src.simulator.HybridSimulator object at 0x000...
Test Day 06,2.060000e+14,85521.049356,4.0,2.060000e+14,-141.530854,58.440666,<src.simulator.HybridSimulator object at 0x000...
Test Day 07,3.024461e+02,72.365509,135.0,9.508057e+01,52.026084,56.802476,<src.simulator.HybridSimulator object at 0x000...
Test Day 08,2.060000e+14,85771.451008,5.0,2.060000e+14,-143.547451,62.696884,<src.simulator.HybridSimulator object at 0x000...
Test Day 09,2.496658e+02,22.364653,132.0,9.530113e+01,51.517084,70.316112,<src.simulator.HybridSimulator object at 0x000...
Test Day 10,2.020000e+14,85683.336767,5.0,2.020000e+14,-154.543877,63.052244,<src.simulator.HybridSimulator object at 0x000...
Test Day 11,2.831291e+02,29.909102,155.0,9.821995e+01,51.405827,58.170603,<src.simulator.HybridSimulator object at 0x000...
Test Day 12,2.050000e+14,85648.262271,5.0,2.050000e+14,-128.257162,72.100116,<src.simulator.HybridSimulator object at 0x000...
Test Day 13,1.920000e+14,85685.359536,4.0,1.920000e+14,-129.383248,71.603313,<src.simulator.HybridSimulator object at 0x000...
